In [1]:
import duckdb
import os
import pandas as pd
import subprocess
from datetime import datetime

db_path = os.path.expanduser('~/trading_infra/storage/trading.duckdb')

# Simple connection - let DuckDB handle it
try:
    conn = duckdb.connect(db_path, read_only=False)
except:
    # If write fails, just read
    conn = duckdb.connect(db_path)

print("="*70)
print("ACCUMULATION TRACKER - Daily Trading Signals")
print("="*70)
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


ACCUMULATION TRACKER - Daily Trading Signals
Generated: 2025-11-10 18:07:25



In [2]:
# CELL 2
print("="*70)
print("HIGH-PROBABILITY CANDIDATES")
print("="*70)

candidates = conn.execute("""
    SELECT 
        ticker,
        signal_strength,
        MAX(volume_ratio) as volume_ratio,
        anomaly_type
    FROM volume_anomalies
    WHERE DATE(timestamp) = CURRENT_DATE
      AND signal_strength IN ('strong', 'moderate')
    GROUP BY ticker, signal_strength, anomaly_type
    ORDER BY MAX(volume_ratio) DESC
""").fetch_df()

if not candidates.empty:
    print(f"\n✓ {len(candidates)} candidates found:\n")
    print(candidates.to_string(index=False))
else:
    print("\n⚠️  No candidates found")

print(f"\nCandidates DF length: {len(candidates)}")


HIGH-PROBABILITY CANDIDATES

✓ 4 candidates found:

ticker signal_strength  volume_ratio anomaly_type
   SPY          strong           2.8        stock
   XLK          strong           2.6        stock
   QQQ        moderate           2.1        stock
   XLF        moderate           1.9        stock

Candidates DF length: 4


In [3]:
# CELL 3 - IMPROVED PLUTUS PROMPT
import yfinance as yf

print("\n" + "="*70)
print("PLUTUS: TRADING SIGNALS (with current market data)")
print("="*70)

if not candidates.empty:
    # Get current prices
    tickers_list = candidates['ticker'].tolist()
    current_prices = {}
    
    for ticker in tickers_list:
        try:
            data = yf.Ticker(ticker)
            price = data.history(period='1d')['Close'].iloc[-1]
            current_prices[ticker] = round(price, 2)
        except:
            current_prices[ticker] = "N/A"
    
    # Build enriched prompt
    candidates_with_prices = candidates.copy()
    candidates_with_prices['current_price'] = candidates_with_prices['ticker'].map(current_prices)
    
    candidates_text = candidates_with_prices.to_string(index=False)
    
    prompt = f"""You are generating trading signals. Use CURRENT PRICES for strike recommendations.

TODAY'S ANOMALIES:
{candidates_text}

For EACH ticker:
1. Recommend ATM or slightly OTM call spreads (based on current_price)
2. Use strikes within 2-5% of current price
3. Target 0DTE or 1DTE expiration
4. Risk/reward ratio (aim for 3:1 minimum)
5. Confidence level (1-10)

Example format:
SPY (current: $580.50) → 580/585 call spread, Risk $200 for $600 reward, Confidence 8/10

Keep responses actionable and brief."""

    print("\nQuerying Plutus with current market data...\n")
    
    try:
        result = subprocess.run(
            ['ollama', 'run', '0xroyce/plutus'],
            input=prompt.encode(),
            capture_output=True,
            timeout=120
        )
        
        analysis = result.stdout.decode('utf-8', errors='ignore')
        print("="*70)
        print(analysis)
        print("="*70)
        
    except subprocess.TimeoutExpired:
        print("⚠️  Plutus timeout")
    except Exception as e:
        print(f"⚠️  Error: {str(e)}")
else:
    print("\n⚠️  No candidates - cannot generate signals")

conn.close()
print("\n✓ Complete")



PLUTUS: TRADING SIGNALS (with current market data)

Querying Plutus with current market data...

⚠️  Plutus timeout

✓ Complete


In [4]:
import duckdb
import os
from datetime import datetime

db_path = os.path.expanduser('~/trading_infra/storage/trading.duckdb')
conn = duckdb.connect(db_path)

# CLEAR old bad data first
conn.execute("DELETE FROM volume_anomalies WHERE DATE(timestamp) = CURRENT_DATE")

print("Creating corrected test anomalies...\n")

# Insert with CORRECT column order: ticker, timestamp, avg_vol, current_vol, ratio, anomaly_type, signal_strength
test_anomalies = [
    ('SPY', 50000000, 140000000, 2.8, 'stock', 'strong'),
    ('QQQ', 45000000, 94500000, 2.1, 'stock', 'moderate'),
    ('XLK', 30000000, 78000000, 2.6, 'stock', 'strong'),
    ('XLF', 25000000, 47500000, 1.9, 'stock', 'moderate'),
]

for ticker, avg_vol, current_vol, ratio, atype, strength in test_anomalies:
    conn.execute("""
        INSERT INTO volume_anomalies
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, [ticker, datetime.now(), avg_vol, current_vol, ratio, atype, strength])
    print(f"✓ {ticker}: {ratio}x ({strength})")

conn.commit()

count = conn.execute("""
    SELECT COUNT(*) FROM volume_anomalies 
    WHERE DATE(timestamp) = CURRENT_DATE
      AND signal_strength IN ('strong', 'moderate')
""").fetchone()[0]

print(f"\n✓ Corrected anomalies: {count}")

conn.close()


Creating corrected test anomalies...

✓ SPY: 2.8x (strong)
✓ QQQ: 2.1x (moderate)
✓ XLK: 2.6x (strong)
✓ XLF: 1.9x (moderate)

✓ Corrected anomalies: 4


In [5]:
# Cell 6: Trading Workflow Reminder
print("\n" + "="*70)
print("TODAY'S TRADING WORKFLOW")
print("="*70)

workflow = """
✓ IF candidates found above:
  1. ✅ Read Plutus analysis (this cell output)
  2. ✅ Copy top candidate ticker
  3. ✅ Open TradingView → search ticker
  4. ✅ Confirm on daily chart: support level, volume profile
  5. ✅ Check 15m/5m: find pullback entry zone
  6. ✅ Open Webull → enter bull call spread
  7. ✅ Risk $100-500 per trade (start small)
  8. ✅ Log trade in execution/trade_log.md + git commit

✓ IF no candidates:
  • Wait for next snapshot
  • Check again in 2 hours
  • No forced trades - only when signal is clear

✓ END OF DAY:
  • Review all trades in trade_log.md
  • Note what worked, what didn't
  • Prepare for next trading day
"""

print(workflow)



TODAY'S TRADING WORKFLOW

✓ IF candidates found above:
  1. ✅ Read Plutus analysis (this cell output)
  2. ✅ Copy top candidate ticker
  3. ✅ Open TradingView → search ticker
  4. ✅ Confirm on daily chart: support level, volume profile
  5. ✅ Check 15m/5m: find pullback entry zone
  6. ✅ Open Webull → enter bull call spread
  7. ✅ Risk $100-500 per trade (start small)
  8. ✅ Log trade in execution/trade_log.md + git commit

✓ IF no candidates:
  • Wait for next snapshot
  • Check again in 2 hours
  • No forced trades - only when signal is clear

✓ END OF DAY:
  • Review all trades in trade_log.md
  • Note what worked, what didn't
  • Prepare for next trading day



In [6]:
# Cell 7: Close Connection
conn.close()
print("\n✓ Accumulation Tracker Complete - Ready to Trade")



✓ Accumulation Tracker Complete - Ready to Trade


In [7]:
import duckdb
import os
from datetime import datetime

db_path = os.path.expanduser('~/trading_infra/storage/trading.duckdb')
conn = duckdb.connect(db_path)

print("Manually detecting anomalies with adjusted threshold...\n")

# Get all tickers
tickers = conn.execute("SELECT DISTINCT ticker FROM sector_etfs").fetchall()

for (ticker,) in tickers:
    # Get last 20 days volume average and latest volume
    result = conn.execute(f"""
        WITH recent AS (
            SELECT volume, timestamp
            FROM sector_etfs
            WHERE ticker = '{ticker}' AND timeframe = '1d'
            ORDER BY timestamp DESC
            LIMIT 21
        )
        SELECT 
            (SELECT volume FROM recent ORDER BY timestamp DESC LIMIT 1) as current_vol,
            AVG(volume) as avg_20d,
            (SELECT volume FROM recent ORDER BY timestamp DESC LIMIT 1) / NULLIF(AVG(volume), 0) as ratio
        FROM (SELECT volume FROM recent ORDER BY timestamp DESC OFFSET 1)
    """).fetchone()
    
    if result:
        current_vol, avg_20d, ratio = result
        if ratio and ratio > 1.3:  # Lower threshold from 1.5 to 1.3
            strength = 'strong' if ratio > 2.5 else 'moderate' if ratio > 1.8 else 'weak'
            
            conn.execute("""
                INSERT OR IGNORE INTO volume_anomalies
                VALUES (?, ?, ?, ?, ?, 'stock', ?)
            """, [ticker, datetime.now(), int(avg_20d or 0), int(current_vol or 0), float(ratio or 0), strength])
            
            print(f"✓ {ticker}: {ratio:.2f}x volume ({strength})")

conn.commit()

# Check results
anomaly_check = conn.execute("""
    SELECT COUNT(*) FROM volume_anomalies
    WHERE DATE(timestamp) = CURRENT_DATE
""").fetchone()[0]

print(f"\n✓ Total anomalies detected today: {anomaly_check}")

conn.close()


Manually detecting anomalies with adjusted threshold...


✓ Total anomalies detected today: 4
